**IMPORTS**

In [1]:
import json
from glob import glob
import pandas as pd
from scipy.io import arff

## Description and understanding of Features for the polish dataset

POLISH_BANKRUPTCY_COLS = {
    'Attr1':  'net profit/ total assets',
    'Attr2':  'total liabilities / total assets',
    'Attr3':  'working capital / total assets',
    'Attr4':  'current assets / short-term liabilities',
    'Attr5':  'cash short-term securities receivables-short-term liabilities / operating expenses-depreciation *365',
    'Attr6':  'retained earnings / total assets',
    'Attr7':  'EBIT / total assets',
    'Attr8':  'book value of equity / total liabilities',
    'Attr9':  'sales / total assets',
    'Attr10': 'equity / total assets',
    'Attr11': 'gross profit+extraordinary items+financial expenses / total assets',
    'Attr12': 'gross profit / short-term liabilities',
    'Attr13': 'gross profit+depreciation / sales',
    'Attr14': 'gross profit+interest / total assets',
    'Attr15': 'total liabilities*365 / gross profit+depreciation',
    'Attr16': 'gross profit+depreciation / total liabilities',
    'Attr17': 'total assets / total liabilities',
    'Attr18': 'gross profit / total assets',
    'Attr19': 'gross profit / sales',
    'Attr20': 'inventory*365 / sales',
    'Attr21': 'sales n / sales n-1',
    'Attr22': 'profit on operating activities / total assets',
    'Attr23': 'net profit / sales',
    'Attr24': 'gross profit 3years / total assets',
    'Attr25': 'equity-share capital / total assets',
    'Attr26': 'net profit+depreciation / total liabilities',
    'Attr27': 'profit on operating activities / financial expenses',
    'Attr28': 'working capital / fixed assets',
    'Attr29': 'logarithm of total assets',
    'Attr30': 'total liabilities-cash / sales',
    'Attr31': 'gross profit+interest / sales',
    'Attr32': 'current liabilities*365 / cost of products sold',
    'Attr33': 'operating expenses / short-term liabilities',
    'Attr34': 'operating expenses / total liabilities',
    'Attr35': 'profit on sales / total assets',
    'Attr36': 'total sales / total assets',
    'Attr37': 'current assets-inventories / long-term liabilities',
    'Attr38': 'constant capital / total assets',
    'Attr39': 'profit on sales / sales',
    'Attr40': 'current assets-inventory-receivables / short-term liabilities',
    'Attr41': 'total liabilities / profit on operating activities+depreciation *12/365',
    'Attr42': 'profit on operating activities / sales',
    'Attr43': 'rotation receivables+inventory turnover in days',
    'Attr44': 'receivables*365 / sales',
    'Attr45': 'net profit / inventory',
    'Attr46': 'current assets-inventory / short-term liabilities',
    'Attr47': 'inventory*365 / cost of products sold',
    'Attr48': 'EBITDA / total assets',
    'Attr49': 'EBITDA / sales',
    'Attr50': 'current assets / total liabilities',
    'Attr51': 'short-term liabilities / total assets',
    'Attr52': 'short-term liabilities*365 / cost of products sold',
    'Attr53': 'equity / fixed assets',
    'Attr54': 'constant capital / fixed assets',
    'Attr55': 'working capital',
    'Attr56': 'sales-cost of products sold / sales',
    'Attr57': 'current assets-inventory-short-term liabilities / sales-gross profit-depreciation',
    'Attr58': 'total costs / total sales',
    'Attr59': 'long-term liabilities / equity',
    'Attr60': 'sales / inventory',
    'Attr61': 'sales / receivables',
    'Attr62': 'short-term liabilities*365 / sales',
    'Attr63': 'sales / short-term liabilities',
    'Attr64': 'sales / fixed assets',
    'class':  ''
}

## Loading data

In [2]:
frames = []
files = sorted(glob("data/polish+companies+bankruptcy+data/*year.arff"))

for file in files:
    data, meta = arff.loadarff(file)
    df = pd.DataFrame(data, columns=meta.names())
    frames.append(df)

df = pd.concat(frames, ignore_index=True)

In [3]:
print("df_shape:",df.shape)

df_shape: (43405, 65)


In [4]:
df.head()

,Attr1,Attr2,Attr3,Attr4,Attr5,Attr6,Attr7,Attr8,Attr9,Attr10,...,Attr56,Attr57,Attr58,Attr59,Attr60,Attr61,Attr62,Attr63,Attr64,class
0,0.200550,0.37951,0.39641,2.0472,32.3510,0.38825,0.249760,1.33050,1.1389,0.50494,...,0.121960,0.39718,0.87804,0.001924,8.4160,5.1372,82.658,4.4158,7.4277,b'0'
1,0.209120,0.49988,0.47225,1.9447,14.7860,0.00000,0.258340,0.99601,1.6996,0.49788,...,0.121300,0.42002,0.85300,0.000000,4.1486,3.2732,107.350,3.4000,60.9870,b'0'
2,0.248660,0.69592,0.26713,1.5548,-1.1523,0.00000,0.309060,0.43695,1.3090,0.30408,...,0.241140,0.81774,0.76599,0.694840,4.9909,3.9510,134.270,2.7185,5.2078,b'0'
3,0.081483,0.30734,0.45879,2.4928,51.9520,0.14988,0.092704,1.86610,1.0571,0.57353,...,0.054015,0.14207,0.94598,0.000000,4.5746,3.6147,86.435,4.2228,5.5497,b'0'
4,0.187320,0.61323,0.22960,1.4063,-7.3128,0.18732,0.187320,0.63070,1.1559,0.38677,...,0.134850,0.48431,0.86515,0.124440,6.3985,4.3158,127.210,2.8692,7.8980,b'0'


## Addressing the Issue of classes to foster multi-classification
From the above the target is the the class column and we can transform it to classes 0 and 1.
To foster multi-clasification, we shall create another target from the class column(tartget), which depends on the years and  prior Knowledge from the dataset Documentation.

In [5]:
frames = []
files = sorted(glob("data/polish+companies+bankruptcy+data/*year.arff"))

for file in files:
    year_num = int(file.split("/")[-1].replace("year.arff", ""))
    horizon = 6 - year_num
    data, meta = arff.loadarff(file)
    df = pd.DataFrame(data, columns=meta.names())
    df["class"] = df["class"].apply(lambda x: int(x.decode().strip("'")))
    df["bankrupt_after_years"] = df["class"].apply(lambda x: horizon if x == 1 else 0)
    
    frames.append(df)

poland_df = pd.concat(frames, ignore_index=True)

In [6]:
poland_df.shape

(43405, 66)

In [7]:
poland_df.head()

,Attr1,Attr2,Attr3,Attr4,Attr5,Attr6,Attr7,Attr8,Attr9,Attr10,...,Attr57,Attr58,Attr59,Attr60,Attr61,Attr62,Attr63,Attr64,class,bankrupt_after_years
0,0.200550,0.37951,0.39641,2.0472,32.3510,0.38825,0.249760,1.33050,1.1389,0.50494,...,0.39718,0.87804,0.001924,8.4160,5.1372,82.658,4.4158,7.4277,0,0
1,0.209120,0.49988,0.47225,1.9447,14.7860,0.00000,0.258340,0.99601,1.6996,0.49788,...,0.42002,0.85300,0.000000,4.1486,3.2732,107.350,3.4000,60.9870,0,0
2,0.248660,0.69592,0.26713,1.5548,-1.1523,0.00000,0.309060,0.43695,1.3090,0.30408,...,0.81774,0.76599,0.694840,4.9909,3.9510,134.270,2.7185,5.2078,0,0
3,0.081483,0.30734,0.45879,2.4928,51.9520,0.14988,0.092704,1.86610,1.0571,0.57353,...,0.14207,0.94598,0.000000,4.5746,3.6147,86.435,4.2228,5.5497,0,0
4,0.187320,0.61323,0.22960,1.4063,-7.3128,0.18732,0.187320,0.63070,1.1559,0.38677,...,0.48431,0.86515,0.124440,6.3985,4.3158,127.210,2.8692,7.8980,0,0


## Saving the model

In [8]:
# Saving the poland_df to csv
poland_df.to_csv("data/poland_df.csv", index=False)